# Introduction to LangChain and LangGraph

**Maryam Berijanian, Dr. Dirk Colbry**  
_Last update: 2026-11-22_

---

## 🛑⚠️ Please save a copy before starting to edit this notebook! ⚠️🛑

Go to **`File -> Save a copy in Drive`** so your edits and exercise solutions are saved to your own Google account.

### How This Interactive Workshop Works
To make this hands-on rather than a passive demo, this notebook includes four interactive elements:
1. **🎛️ Personalized Form Cells (`#@param`)**: Customize prompts with your own interests, hometown, and courses so your outputs are uniquely yours.
2. **🧩 Hinge-Line Micro-Puzzles (`TODO`)**: Short 1–2 line coding challenges where you wire critical agent pieces, followed by automated checks and collapsed **💡 Solution** cells if you get stuck.
3. **🔴 Red-Team Mini-Challenges**: Quick 2-minute challenges to test—and intentionally break—agent routers and critics.
4. **🎓 Capstone Exercise**: Build a **University Course Survival Copilot** combining RAG, custom grade-math tools, and Human-in-the-Loop email approval.

# 1) Setup for Google Colab

**Note: Google Colab Pro is free for students and educators! Sign up [here](https://colab.research.google.com/signup).**

### Three Ways to Run Models in This Workshop

| Provider | Cost | Hardware Needed | Native Tool Calling | Setup Time |
| :--- | :--- | :--- | :--- | :--- |
| **1. `gemini` (Recommended)** | **Free Tier** | Standard CPU Runtime | ✅ Yes | ~30 seconds |
| **2. `huggingface` (Local Open Model)** | **Free** | Free **T4 GPU** Runtime | ⚠️ Limited (Text/RAG/Graphs) | ~45 seconds (download) |
| **3. `openai` (Optional)** | Paid API | Standard CPU Runtime | ✅ Yes | ~30 seconds |

#### Quick Setup for Option 1: Free Gemini API Key (Recommended)
1. Click the **🔑 (Secrets)** icon in the left sidebar of Google Colab.
2. Click **"Import key from Google AI Studio"** (or generate a free key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey) — no credit card required).
3. Name the secret **`GEMINI_API_KEY`** and toggle **Notebook access** ON.

#### Quick Setup for Option 2: Zero-Key Local Hugging Face Model
- Uses `Qwen/Qwen2.5-3B-Instruct` (ungated — **no `HF_TOKEN` required!**).
- Go to **`Runtime -> Change runtime type -> T4 GPU`** before running the cells below.
- *Note:* Sections 2, 3, 4, 6, and 7 run locally on `huggingface`, whereas Sections 5 and 9 use native `.bind_tools()`, which works best with `gemini` or `openai`.

In [ ]:
!pip install -qU \
  langgraph \
  langchain \
  langchain-core \
  langchain-community \
  langchain-google-genai \
  langchain-huggingface \
  langchain-openai \
  accelerate \
  faiss-cpu \
  sentence-transformers \
  wikipedia

In [ ]:
# @title 1.1 Select & Initialize Model Provider
import os
import torch
from google.colab import userdata

MODEL_PROVIDER = "gemini"  # @param ["gemini", "huggingface", "openai"]

print(f"Initializing workshop with provider: {MODEL_PROVIDER.upper()}...")

if MODEL_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI

    try:
        gemini_key = userdata.get("GEMINI_API_KEY")
    except Exception as e:
        raise RuntimeError(
            "Missing GEMINI_API_KEY! Open the 🔑 Secrets tab in the left Colab sidebar, "
            "add 'GEMINI_API_KEY' (free via Google AI Studio), and toggle Notebook access ON."
        ) from e

    os.environ["GOOGLE_API_KEY"] = gemini_key
    active_chat = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.2,
    )
    print("✅ Loaded: Gemini 2.5 Flash (CPU-friendly, Native Tool Calling enabled)")

elif MODEL_PROVIDER == "huggingface":
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
    from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

    MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
    if not torch.cuda.is_available():
        print("⚠️ WARNING: Running on CPU will be slow! Switch to Runtime -> Change runtime type -> T4 GPU.")

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )

    gen_pipe = pipeline(
        task="text-generation",
        model=mdl,
        tokenizer=tok,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        return_full_text=False,
    )
    active_chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=gen_pipe))
    print(f"✅ Loaded: {MODEL_NAME} | Device: {mdl.device}")

elif MODEL_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    active_chat = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    print("✅ Loaded: OpenAI gpt-4o-mini")

# 2) Basic Usage: Messages & Personalized Invocation

LangChain standardizes chat interactions across model providers using structured message objects:
- **`SystemMessage`**: Sets the assistant's persona, boundaries, or behavioral rules.
- **`HumanMessage`**: Represents user input.
- **`AIMessage`**: Represents the model's response (including text and optional tool calls).

Use the form fields below to test `active_chat.invoke()` on a topic of your choice!

In [ ]:
# @title 2.1 Run Your First Personalized LLM Call
from langchain_core.messages import SystemMessage, HumanMessage

your_hobby_or_topic = "baking sourdough bread"  # @param {type:"string"}
persona_style = "witty science communicator"  # @param ["concise coach", "witty science communicator", "dramatic sports commentator", "friendly pirate"]

messages = [
    SystemMessage(content=f"You are a {persona_style}. Keep answers crisp and engaging."),
    HumanMessage(content=f"Give me a 2-step beginner plan to get started with: {your_hobby_or_topic}."),
]

resp = active_chat.invoke(messages)
print(resp.content)

# 3) Sequential Prompt Chaining & LCEL

Before building cyclic graphs or autonomous agents, the simplest multi-step pattern is **Sequential Prompt Chaining**: breaking a task into specialized steps where the output of Step $k$ becomes the input to Step $k+1$.

Below, three specialized role functions collaborate sequentially to invent a character for a custom setting.

In [ ]:
# @title 3.1 Three-Step Sequential Character Chain
story_setting = "a deep-sea research station"  # @param {type:"string"}

def step_1_inventor(setting: str) -> str:
    prompt = f"Invent ONLY ONE creative first name for someone working in {setting}. Output just the single name."
    return active_chat.invoke(prompt).content.strip()

def step_2_trait(name: str, setting: str) -> str:
    prompt = f"Write ONLY ONE short sentence stating {name}'s unusual favorite habit while living in {setting}."
    return active_chat.invoke(prompt).content.strip()

def step_3_backstory(sentence: str) -> str:
    prompt = f"Expand this sentence by adding ONLY one short, surprising reason why at the end: '{sentence}'"
    return active_chat.invoke(prompt).content.strip()

print(f"--- Starting Chain (Setting: {story_setting}) ---\n")

name_out = step_1_inventor(story_setting)
print(f"Step 1 (Name):\n{name_out}\n")

trait_out = step_2_trait(name_out, story_setting)
print(f"Step 2 (Habit):\n{trait_out}\n")

final_story = step_3_backstory(trait_out)
print(f"Step 3 (Expanded Story):\n{final_story}")

### LangChain Expression Language (LCEL)

Writing manual wrapper functions gets repetitive fast. **LangChain Expression Language (LCEL)** lets you declaratively pipe components (`Prompt -> Model -> OutputParser`) together using the `|` operator.

Every LCEL pipeline automatically supports `.invoke()`, `.batch()`, and real-time token `.stream()`.

In [ ]:
# @title 3.2 Compose & Stream an LCEL Pipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

concept_to_explain = "multi-agent systems"  # @param {type:"string"}
target_audience = "a curious 10-year-old"  # @param ["a curious 10-year-old", "a software engineer", "a university dean"]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain technical concepts clearly for {audience}."),
    ("human", "In exactly two sentences, explain what {concept} is and why it is useful.")
])

# Declarative LCEL pipeline using the pipe (|) operator
lcel_chain = prompt | active_chat | StrOutputParser()

print("Streaming output token-by-token:\n")
for chunk in lcel_chain.stream({"concept": concept_to_explain, "audience": target_audience}):
    print(chunk, end="", flush=True)
print()

# 4) Routing vs. Chaining in LangGraph

While linear chains execute a fixed sequence (`Plan -> Research -> Write`), real-world workflows need **dynamic control flow** where the next node depends on runtime state:

- **Linear Chaining**: `START -> plan -> research -> write -> END` (wastes latency/tokens researching trivial prompts).
- **Conditional Routing**: `START -> classify -> plan -> (skip research if trivial, else research) -> write -> END`.

In **LangGraph**:
1. We define a shared **`TypedDict` State** schema.
2. Each **Node** is a Python function taking the current `state` and returning **only the updated keys** (a state delta).
3. **`add_conditional_edges`** inspects the state and dynamically chooses the next node.

In [ ]:
from typing import TypedDict, Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

class ResearchState(TypedDict, total=False):
    task: str
    label: str
    plan: str
    facts: str
    draft: str

ClassifyP = ChatPromptTemplate.from_template(
    "Evaluate whether the following task requires looking up external/current real-world facts "
    "or if it is a self-contained/trivial request (like basic math, greetings, or simple creative phrasing).\n\n"
    "TASK: {task}\n\n"
    "Output ONLY one word: 'research_needed' or 'trivial'."
)

PlanP = ChatPromptTemplate.from_template(
    "Create a concise 2-step numbered plan to answer the task.\n"
    "TASK: {task}\n"
    "Output ONLY the 2 numbered steps."
)

FactsP = ChatPromptTemplate.from_template(
    "Provide 1-2 short factual sentences from your internal knowledge to help answer the task.\n"
    "TASK: {task}\nPLAN:\n{plan}\n\n"
    "If answering accurately requires live internet data or post-cutoff events you cannot verify, "
    "output EXACTLY: INSUFFICIENT_FACTS"
)

WriteP = ChatPromptTemplate.from_template(
    "Write a concise final response to the task.\n"
    "TASK: {task}\nFACTS: {facts}\n\n"
    "If FACTS equals 'INSUFFICIENT_FACTS', state clearly that live external search tools are required to answer accurately."
)

# Notice how LangGraph nodes return ONLY the state delta (updated keys)
def classify_node(s: ResearchState) -> dict:
    raw = (ClassifyP | active_chat | StrOutputParser()).invoke({"task": s["task"]}).strip().lower()
    label = "trivial" if "trivial" in raw else "research_needed"
    return {"label": label}

def plan_node(s: ResearchState) -> dict:
    plan = (PlanP | active_chat | StrOutputParser()).invoke({"task": s["task"]}).strip()
    return {"plan": plan}

def research_node(s: ResearchState) -> dict:
    facts = (FactsP | active_chat | StrOutputParser()).invoke({
        "task": s["task"],
        "plan": s.get("plan", ""),
    }).strip()
    return {"facts": facts}

def write_node(s: ResearchState) -> dict:
    facts = s.get("facts", "Not needed (trivial self-contained task — answer directly).")
    draft = (WriteP | active_chat | StrOutputParser()).invoke({
        "task": s["task"],
        "facts": facts,
    }).strip()
    return {"draft": draft}

def route_after_plan(s: ResearchState) -> Literal["skip_to_write", "do_research"]:
    return "skip_to_write" if s.get("label") == "trivial" else "do_research"

print("✅ ResearchState, prompts, and node functions defined.")

In [ ]:
# ==============================================================================
# 🧩 PUZZLE #1: Wire the Conditional Edge in `g_route`
# ==============================================================================

# 1. Reference: Pure Linear Chain Graph (Always runs research)
g_chain = StateGraph(ResearchState)
g_chain.add_node("plan", plan_node)
g_chain.add_node("research", research_node)
g_chain.add_node("write", write_node)

g_chain.add_edge(START, "plan")
g_chain.add_edge("plan", "research")
g_chain.add_edge("research", "write")
g_chain.add_edge("write", END)
app_chain = g_chain.compile()

# 2. Your Turn: Conditional Routed Graph
g_route = StateGraph(ResearchState)
g_route.add_node("classify", classify_node)
g_route.add_node("plan", plan_node)
g_route.add_node("research", research_node)
g_route.add_node("write", write_node)

g_route.add_edge(START, "classify")
g_route.add_edge("classify", "plan")

# TODO (Puzzle #1): Add conditional edges starting from "plan" using `route_after_plan`.
# Map "skip_to_write" -> "write" and "do_research" -> "research".
# Replace `None` below with `g_route.add_conditional_edges(...)`:
None

g_route.add_edge("research", "write")
g_route.add_edge("write", END)
app_route = g_route.compile()

# --- Automated Self-Check ---
test_trivial = app_route.invoke({"task": "What is 12 + 15?"})
assert "draft" in test_trivial, "❌ Graph reached END before 'write'! Did you wire add_conditional_edges from 'plan'?"
assert "facts" not in test_trivial, "❌ Trivial task still ran 'research'! Check your conditional mapping."
print("✅ Puzzle #1 Passed! Trivial tasks bypass research while complex tasks run research.")

In [ ]:
# @title 💡 Click to Show/Run Solution (Puzzle #1)
g_route = StateGraph(ResearchState)
g_route.add_node("classify", classify_node)
g_route.add_node("plan", plan_node)
g_route.add_node("research", research_node)
g_route.add_node("write", write_node)

g_route.add_edge(START, "classify")
g_route.add_edge("classify", "plan")
g_route.add_conditional_edges(
    "plan",
    route_after_plan,
    {"skip_to_write": "write", "do_research": "research"},
)
g_route.add_edge("research", "write")
g_route.add_edge("write", END)
app_route = g_route.compile()
print("✅ Solution compiled into `app_route`!")

In [ ]:
# @title 4.1 Compare Chain vs. Router (+ 🔴 2-Minute Red-Team Challenge!)
# 🔴 RED-TEAM CHALLENGE: Can you craft a `user_task` that genuinely asks about a live/current event,
# yet tricks `classify_node` into labeling it "trivial" so it skips research?

user_task = "What are the top fashion trends right now in late 2026?"  # @param {type:"string"}

s_chain = app_chain.invoke({"task": user_task})
print("=== 1. LINEAR CHAIN ===")
print("PLAN:\n", s_chain.get("plan"))
print("FACTS:\n", s_chain.get("facts"))
print("DRAFT:\n", s_chain.get("draft"), "\n")

s_route = app_route.invoke({"task": user_task})
print("=== 2. ROUTED GRAPH ===")
print("ROUTER LABEL:", s_route.get("label"))
print("PLAN:\n", s_route.get("plan"))
print("FACTS:\n", s_route.get("facts", "(Skipped research node!)"))
print("DRAFT:\n", s_route.get("draft"))

# 5) Tool Use & Agentic ReAct Loops

In Section 4, our agent hit a wall on live questions because it had no access to the outside world. **[Tools](https://python.langchain.com/docs/concepts/tools/)** let LLMs call Python functions, live APIs, calculators, or databases.

### How Tool Calling Works in LangChain & LangGraph
1. **Define with `@tool`**: Turns a typed, documented Python function into a structured JSON tool schema. The LLM reads the **function name, parameter types, and docstring** to decide *when* and *how* to call it!
2. **Bind with `.bind_tools(tools)`**: Exposes tool schemas to the chat model (`tool_choice` can optionally force a specific tool).
3. **Automate Loops with `ToolNode` + `tools_condition`**: Instead of writing messy `if/else` blocks to execute tools manually, LangGraph's prebuilt `ToolNode` executes any requested tool calls (even multiple tools in parallel!) and loops the `ToolMessage` results right back to the LLM.

*(Note: Native `.bind_tools()` requires a tool-capable provider such as `gemini` or `openai`.)*

In [ ]:
import datetime as dt
import requests
import wikipedia
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

# --- Tool 1: Live Open-Meteo Weather API ---
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location (e.g., 42.7325 for East Lansing)")
    longitude: float = Field(..., description="Longitude of the location (e.g., -84.5555 for East Lansing)")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> str:
    """Fetch the current real-time temperature in Celsius for given latitude and longitude coordinates."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1,
        "timezone": "UTC",
    }
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()

    now_utc = dt.datetime.now(dt.timezone.utc)
    times = [
        dt.datetime.fromisoformat(t.replace("Z", "+00:00")) if t.endswith("Z")
        else dt.datetime.fromisoformat(t).replace(tzinfo=dt.timezone.utc)
        for t in data["hourly"]["time"]
    ]
    temps = data["hourly"]["temperature_2m"]
    idx = min(range(len(times)), key=lambda i: abs(times[i] - now_utc))
    return f"Current temperature near ({latitude:.3f}, {longitude:.3f}) is {temps[idx]}°C (UTC hour: {times[idx].strftime('%H:%M')})."

# --- Tool 2: Wikipedia Search API ---
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia and return short page summaries for historical facts, books, people, or places."""
    try:
        titles = wikipedia.search(query)
    except Exception as e:
        return f"Wikipedia search error: {e}"
    if not titles:
        return "No Wikipedia pages found."

    summaries = []
    for title in titles[:2]:
        try:
            page = wikipedia.page(title=title, auto_suggest=False)
            summaries.append(f"Title: {title}\nSummary: {page.summary[:600]}")
        except Exception:
            continue
    return "\n\n".join(summaries) if summaries else "No matching Wikipedia summary found."

print("🔍 Inspected JSON Tool Schema sent to the LLM:")
print(convert_to_openai_tool(get_current_temperature))
print("\nDirect Tool Test:", get_current_temperature.invoke({"latitude": 42.7325, "longitude": -84.5555}))

In [ ]:
# @title 5.1 Build & Run a Multi-Tool LangGraph Agent
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

tools = [get_current_temperature, search_wikipedia]
llm_with_tools = active_chat.bind_tools(tools)
# Tip: You can also force a specific tool via `active_chat.bind_tools(tools, tool_choice="search_wikipedia")`

def assistant_node(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

agent_builder = StateGraph(MessagesState)
agent_builder.add_node("assistant", assistant_node)
agent_builder.add_node("tools", ToolNode(tools))

agent_builder.add_edge(START, "assistant")
# Routes to "tools" if the model emitted tool_calls; otherwise routes to END
agent_builder.add_conditional_edges("assistant", tools_condition)
agent_builder.add_edge("tools", "assistant")  # Loop tool outputs back to the LLM!

tool_agent = agent_builder.compile()

multi_tool_question = "What year was Pride and Prejudice originally published, and what is the temperature right now in East Lansing, Michigan?"  # @param {type:"string"}

result = tool_agent.invoke({"messages": [("user", multi_tool_question)]})
for msg in result["messages"]:
    msg.pretty_print()

In [ ]:
# ==============================================================================
# 🧩 PUZZLE #2: Fix "The Blind Tool Docstring"
# ==============================================================================
# Why did the LLM know how to use Weather and Wikipedia above? ONLY because of their docstrings & parameter names!
# Below is a working Python function with a terrible docstring and vague parameter names.
# Run this cell as-is first to see what happens, then fix the docstring (and/or arg names) so the test passes!

@tool
def split_check_tool(x: float, y: float, z: int) -> str:
    # TODO (Puzzle #2): Rewrite this docstring clearly so the LLM knows what x, y, and z mean
    # and WHEN to call this tool!
    """Does some arithmetic."""
    per_person = (x * (1.0 + y / 100.0)) / z
    return f"Each of the {z} people owes exactly ${per_person:.2f} (including {y}% tip)."

puzzle_llm = active_chat.bind_tools([split_check_tool])

def bill_assistant(state: MessagesState):
    sys = SystemMessage(content="Use available tools whenever an exact calculation tool applies; otherwise answer normally.")
    return {"messages": [puzzle_llm.invoke([sys] + state["messages"])]}

g_bill = StateGraph(MessagesState)
g_bill.add_node("assistant", bill_assistant)
g_bill.add_node("tools", ToolNode([split_check_tool]))
g_bill.add_edge(START, "assistant")
g_bill.add_conditional_edges("assistant", tools_condition)
g_bill.add_edge("tools", "assistant")
bill_agent = g_bill.compile()

bill_res = bill_agent.invoke({
    "messages": [("user", "Split a $137.40 restaurant dinner bill with an 18% tip evenly among 5 friends.")]
})

tool_called = any(m.type == "tool" for m in bill_res["messages"])
print("Final Answer:", bill_res["messages"][-1].content)
assert tool_called and "32.43" in bill_res["messages"][-1].content, (
    "❌ The model either ignored `split_check_tool` or passed arguments in the wrong order! "
    "Improve the docstring in `split_check_tool` to explain `x` (bill), `y` (tip %), and `z` (people)."
)
print("✅ Puzzle #2 Passed! The LLM discovered and called `split_check_tool` with the exact arguments.")

In [ ]:
# @title 💡 Click to Show/Run Solution (Puzzle #2)
@tool
def split_check_tool(bill_amount: float, tip_percentage: float, num_people: int) -> str:
    """Calculate how much each person owes when splitting a restaurant bill with a tip percentage among a group of people."""
    per_person = (bill_amount * (1.0 + tip_percentage / 100.0)) / num_people
    return f"Each of the {num_people} people owes exactly ${per_person:.2f} (including {tip_percentage}% tip)."

print("✅ Descriptive parameter names + docstring ready!")

# 6) Reflection via Cyclic Graphs

LLM agents substantially improve output quality through **Reflection**—critiquing and iteratively revising drafts before returning a final answer:

1. **Self-Reflection**: A single model critiques its own draft for clarity or style.
2. **External / Rubric Reflection**: A dedicated critic evaluates the draft against explicit, verifiable constraints (or unit tests) and sends structured failure feedback back to the generator.

Linear chains can only revise once. With **LangGraph cycles** (`generator -> critic -> conditional loop back to generator`), the agent iterates automatically until all rubric rules pass (or `max_iterations` is reached).

In [ ]:
# @title 6.1 Run a Cyclic Generator-Critic Reflection Loop
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

topic_to_explain = "Retrieval-Augmented Generation (RAG)"  # @param {type:"string"}
custom_extra_rule = "Must include a cooking or kitchen analogy."  # @param {type:"string"}

class ReflectionState(TypedDict, total=False):
    topic: str
    extra_rule: str
    draft: str
    critique: str
    iteration: int

DraftP = ChatPromptTemplate.from_template(
    "Write a clear explanation of '{topic}' for a college freshman.\n"
    "Previous draft: {draft}\n"
    "Critic feedback to fix: {critique}\n\n"
    "Output ONLY the revised explanation text (no preambles or quotes)."
)

RubricCritP = ChatPromptTemplate.from_template(
    "You are a strict rubric grader. Check the DRAFT against ALL 3 rules:\n"
    "Rule 1: Must be 35 words or fewer.\n"
    "Rule 2: Must NOT contain unexplained acronyms.\n"
    "Rule 3: {extra_rule}\n\n"
    "DRAFT: {draft}\n\n"
    "If ALL 3 rules are strictly satisfied, output EXACTLY the single word: PASS\n"
    "Otherwise, state concisely which rule(s) failed and how to fix the draft in 1 sentence."
)

def generator_node(s: ReflectionState) -> dict:
    it = s.get("iteration", 0) + 1
    draft = (DraftP | active_chat | StrOutputParser()).invoke({
        "topic": s["topic"],
        "draft": s.get("draft", "(First attempt — write an initial 2-3 sentence explanation)"),
        "critique": s.get("critique", "None yet."),
    }).strip()
    return {"draft": draft, "iteration": it}

def rubric_critic_node(s: ReflectionState) -> dict:
    word_count = len(s["draft"].split())
    fb = (RubricCritP | active_chat | StrOutputParser()).invoke({
        "draft": f"{s['draft']} (Measured word count: {word_count} words)",
        "extra_rule": s["extra_rule"],
    }).strip()
    print(f"--- Iteration {s['iteration']} ({word_count} words) ---")
    print(f"Draft:    {s['draft']}")
    print(f"Critique: {fb}\n")
    return {"critique": fb}

def should_continue_reflecting(s: ReflectionState) -> Literal["revise", "done"]:
    if s["critique"].strip().upper().startswith("PASS") or s["iteration"] >= 3:
        return "done"
    return "revise"

refl_builder = StateGraph(ReflectionState)
refl_builder.add_node("generator", generator_node)
refl_builder.add_node("critic", rubric_critic_node)

refl_builder.add_edge(START, "generator")
refl_builder.add_edge("generator", "critic")
refl_builder.add_conditional_edges(
    "critic",
    should_continue_reflecting,
    {"revise": "generator", "done": END},
)

app_reflect = refl_builder.compile()
final_refl = app_reflect.invoke({"topic": topic_to_explain, "extra_rule": custom_extra_rule, "iteration": 0})

print("✅ Final Approved Output:\n", final_refl["draft"])

# 7) Retrieval-Augmented Generation (RAG)

**Retrieval-Augmented Generation (RAG)** grounds LLM responses in external documents (private PDFs, lab wikis, course syllabi) rather than relying solely on frozen model weights:

1. **Embed**: Convert document chunks into dense semantic vectors using `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`).
2. **Index & Retrieve**: Store vectors in **FAISS** and fetch the top-$k$ nearest neighbors for a user query.
3. **Generate**: Inject ONLY the retrieved passages into the prompt so out-of-scope questions are cleanly refused instead of hallucinated.

In [ ]:
import warnings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

warnings.filterwarnings("ignore", category=FutureWarning)

base_docs = [
    Document(page_content="LangGraph enables stateful, cyclic, multi-agent workflows represented as graphs with nodes and conditional edges."),
    Document(page_content="LCEL (LangChain Expression Language) composes prompts, models, and parsers declaratively using the pipe operator (|)."),
    Document(page_content="Tools extend LLMs beyond text generation by binding structured Python functions via @tool and .bind_tools()."),
    Document(page_content="Reflection loops pair a generator node with a critic node to iteratively verify and refine agent outputs."),
    Document(page_content="FAISS is an efficient similarity-search library for dense vectors commonly used as an in-memory vector store for RAG."),
]

print("Loading sentence-transformer embeddings & indexing FAISS vector store...")
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = FAISS.from_documents(base_docs, emb)
print("✅ Vector store ready!")

RAGPrompt = ChatPromptTemplate.from_template(
    "Answer the question concisely using ONLY the provided context.\n"
    "If the context does not contain the answer, reply: 'I cannot answer based on the provided documents.'\n\n"
    "CONTEXT:\n{ctx}\n\nQUESTION: {q}"
)

def rag_answer(question: str, k: int = 2):
    hits = vs.similarity_search(question, k=k)
    context = "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(hits))
    answer = (RAGPrompt | active_chat | StrOutputParser()).invoke({"ctx": context, "q": question})
    return hits, answer

# Test 1: In-domain question
_, ans_in = rag_answer("How does LangGraph differ from linear chains?")
print("\n[In-Domain Question] ->", ans_in)

# Test 2: Unrelated question (verifying hallucination guard)
_, ans_out = rag_answer("What is the boiling point of mercury at sea level?")
print("[Out-of-Scope Question] ->", ans_out)

In [ ]:
# ==============================================================================
# 🧩 PUZZLE #3: Inject Your Own Secret Knowledge into FAISS
# ==============================================================================
# Create 2 custom `Document` objects containing fictional "secret rules" (e.g., a secret club passcode,
# mascot name, or snack policy) that the LLM could NEVER know from pretraining!

# TODO (Puzzle #3): Add at least 2 Document(page_content="...") items to `my_custom_docs`
# and ask a question about one of your secret facts in `my_secret_question`.
my_custom_docs = [
    # Example: Document(page_content="...")
]
my_secret_question = ""  # <-- Ask about your custom rule here!

assert len(my_custom_docs) >= 2 and my_secret_question.strip(), (
    "❌ Add at least 2 `Document(page_content=...)` items to `my_custom_docs` and set `my_secret_question`!"
)

vs.add_documents(my_custom_docs)
hits, secret_ans = rag_answer(my_secret_question, k=2)

print("Retrieved Top Passages:")
for i, d in enumerate(hits, 1):
    print(f"  {i}. {d.page_content}")
print("\nGrounded Answer:\n", secret_ans)
assert "cannot answer" not in secret_ans.lower(), "❌ RAG couldn't find your answer—check that your question matches your custom Document!"
print("\n✅ Puzzle #3 Passed! Your custom knowledge is live in the vector index.")

In [ ]:
# @title 💡 Click to Show/Run Solution (Puzzle #3)
my_custom_docs = [
    Document(page_content="The Quantum Baking Club meets every Friday at 5:15 PM in Room 404 and requires members to bring cardamom buns."),
    Document(page_content="The emergency Wi-Fi password for the Autonomous Robotics Lab is 'Copper-Badger-99'."),
]
vs.add_documents(my_custom_docs)
hits, secret_ans = rag_answer("What food is required at Quantum Baking Club meetings, and when do they meet?", k=2)
print("Grounded Answer:", secret_ans)

# 8) Human-in-the-Loop (HITL) & Graph Visualization

High-stakes actions—sending emails, modifying databases, executing financial trades, or running external queries—often require **Human-in-the-Loop (HITL)** oversight before execution.

* **Strengths**: Prevents costly agent errors, enforces human accountability, and lets humans steer ambiguous parameters mid-flight.
* **Trade-offs**: Human review does not scale to millions of automated calls and depends on operator attention.

### Modern HITL in LangGraph (`MemorySaver` + `interrupt` + `Command`)
When a graph is compiled with a **checkpointer** (`MemorySaver`), calling `interrupt(...)` inside any node or tool:
1. **Freezes** graph state safely at that exact line under a `thread_id`.
2. Surfaces a review payload to the human user.
3. **Resumes on the exact same `thread_id`** when invoked with `Command(resume=human_input)`.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langgraph.graph import StateGraph, START, END, MessagesState

def reviewed_wikipedia_node(state: MessagesState):
    raw_user_msg = state["messages"][-1].content
    planned_query = raw_user_msg.replace("search:", "").strip()

    # ⏸️ Pause execution mid-node and surface the planned query to the human!
    human_override = interrupt({
        "instruction": "Approve or rewrite the planned Wikipedia search query.",
        "planned_query": planned_query,
    })

    # ▶️ Execution resumes right here on the SAME thread when Command(resume=...) is called!
    final_query = human_override.strip() if human_override.strip() else planned_query
    wiki_context = search_wikipedia.invoke({"query": final_query})

    answer = active_chat.invoke([
        SystemMessage(content="Answer in 2-3 sentences using ONLY the provided Wikipedia summary."),
        HumanMessage(content=f"Topic: {final_query}\n\nWikipedia Summary:\n{wiki_context}")
    ])
    return {"messages": [answer]}

hitl_builder = StateGraph(MessagesState)
hitl_builder.add_node("reviewed_search", reviewed_wikipedia_node)
hitl_builder.add_edge(START, "reviewed_search")
hitl_builder.add_edge("reviewed_search", END)

hitl_app = hitl_builder.compile(checkpointer=MemorySaver())

# --- Step 1: Start run on thread "hitl-demo-1" (will pause at interrupt!) ---
thread_cfg = {"configurable": {"thread_id": "hitl-demo-1"}}
_ = hitl_app.invoke({"messages": [("user", "search: Michigan State University")]}, config=thread_cfg)

snapshot = hitl_app.get_state(thread_cfg)
interrupt_payload = snapshot.tasks[0].interrupts[0].value
print(f"⏸️ Graph Paused at node: {snapshot.next}")
print(f"📋 Planned Query: '{interrupt_payload['planned_query']}'")

# --- Step 2: Human approves (Enter) or types a completely new topic, resuming the SAME thread ---
user_edit = input("Type a new Wikipedia topic to override (or press Enter to keep planned query): ")
resumed = hitl_app.invoke(Command(resume=user_edit), config=thread_cfg)

print("\n✅ Resumed Final Answer:\n", resumed["messages"][-1].content)

In [ ]:
# @title 8.1 Bonus: Visualize Any Graph from Today's Workshop
from IPython.display import display, Image, Markdown

graph_to_visualize = "app_reflect"  # @param ["app_route", "tool_agent", "app_reflect", "hitl_app"]

graph_map = {
    "app_route": app_route,
    "tool_agent": tool_agent,
    "app_reflect": app_reflect,
    "hitl_app": hitl_app,
}
viz = graph_map[graph_to_visualize].get_graph()

try:
    display(Image(viz.draw_mermaid_png()))
except Exception:
    display(Markdown(f"```mermaid\n{viz.draw_mermaid()}\n```"))

# 9) 🎓 Capstone Exercise: Build a "University Course Survival Copilot"

Time to combine **everything** you learned today—**RAG**, **Custom `@tool` Creation**, **`.bind_tools()`**, **LangGraph ReAct Loops**, and **Human-in-the-Loop `interrupt()`**—into an application every student wishes they had during finals week!

### What Your Copilot Can Do:
1. **`lookup_syllabus` (RAG Tool — Provided)**: Searches course syllabus policies (grading weights, late penalties, office hours).
2. **`needed_final_exam_score` (Math Tool — `TODO #1`)**: Calculates the exact percentage needed on the final exam to reach a desired overall course grade.
3. **`send_extension_email` (HITL Tool — Provided)**: Drafts a deadline extension email to the professor and **pauses via `interrupt()`** so you can inspect and approve/cancel before it sends!

*(Note: Requires `MODEL_PROVIDER = "gemini"` or `"openai"` for multi-tool calling.)*

### Complete the 3 `TODO`s in the Cell Below:
- **`TODO #1a & #1b`**: Write a clear docstring for `needed_final_exam_score` and fill in the algebraic formula for `needed`.
- **`TODO #2`**: Bind `capstone_tools` to `active_chat`.
- **`TODO #3`**: Wire the conditional edges from `"copilot"` (`tools_condition`) and loop `"tools"` back to `"copilot"`.

In [ ]:
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

# 1. Course Syllabus Vector Store (Feel free to customize policies!)
syllabus_docs = [
    Document(page_content="Course Grading Breakdown: Homework Assignments = 40%, Midterm Project = 25%, Final Exam = 35%. An overall score of 90.0% or higher guarantees a 4.0 (A)."),
    Document(page_content="Late Submission Policy: Late homework loses 10% per day for up to 3 days. Extension requests must be emailed at least 24 hours before the deadline."),
    Document(page_content="Instructor Office Hours: Tuesdays & Thursdays 2:00 PM - 3:30 PM in Engineering Room 312, or on Zoom by appointment."),
]
syllabus_vs = FAISS.from_documents(syllabus_docs, emb)

# --- TOOL 1: Syllabus RAG Lookup (Provided) ---
@tool
def lookup_syllabus(query: str) -> str:
    """Search the official course syllabus for grading weights, exam percentages, late policies, or office hours."""
    hits = syllabus_vs.similarity_search(query, k=2)
    return "\n".join(d.page_content for d in hits)

# --- TOOL 2: Final Exam Grade Calculator (TODO #1) ---
@tool
def needed_final_exam_score(current_weighted_avg: float, final_exam_weight: float, target_course_grade: float) -> str:
    # TODO #1a: Replace this vague docstring so the LLM knows WHEN to call this tool and what the 3 inputs represent!
    """TODO: Write a clear description of what this tool calculates."""

    # Normalize weight in case the model passes 35 instead of 0.35
    w = final_exam_weight / 100.0 if final_exam_weight > 1.0 else final_exam_weight

    # Hint: overall_grade = current_weighted_avg * (1 - w) + needed * w
    # TODO #1b: Solve for `needed` (the required percentage score on the final exam):
    needed = None  # <-- REPLACE `None` WITH THE FORMULA!

    if needed is None:
        raise NotImplementedError("Complete TODO #1b inside `needed_final_exam_score`!")
    return f"To achieve {target_course_grade}% overall (with final exam weight {w:.0%}), you need {needed:.1f}% on the final exam."

# --- TOOL 3: Extension Email Sender with HITL Approval (Provided) ---
@tool
def send_extension_email(reason: str, extra_days: int) -> str:
    """Draft and send an official homework extension request email to the course professor."""
    email_draft = (
        f"Subject: Homework Extension Request ({extra_days} days)\n"
        f"Dear Professor,\n"
        f"I am writing at least 24 hours in advance per the syllabus policy to request a {extra_days}-day extension "
        f"due to {reason}. Thank you for your time and consideration.\n"
        f"Best regards,\nStudent"
    )
    decision = interrupt({
        "email_draft": email_draft,
        "prompt": "Approve sending this email? Type 'send' to approve or 'cancel' to abort:"
    })
    if decision.strip().lower() == "send":
        return "✅ Extension request email approved by student and sent to the professor!"
    return "❌ Extension request email was canceled by the student."

# --- Build the Course Survival Copilot Graph ---
capstone_tools = [lookup_syllabus, needed_final_exam_score, send_extension_email]

# TODO #2: Bind `capstone_tools` to `active_chat`
capstone_llm = None  # <-- REPLACE `None` USING `active_chat.bind_tools(...)`

def copilot_node(state: MessagesState):
    if capstone_llm is None:
        raise NotImplementedError("Complete TODO #2 by binding `capstone_tools` to `active_chat`!")
    sys = SystemMessage(
        content=(
            "You are a supportive University Course Survival Copilot. "
            "Always check `lookup_syllabus` first if you need course policies or exam weights, "
            "use `needed_final_exam_score` for grade math, and use `send_extension_email` when asked to request an extension."
        )
    )
    return {"messages": [capstone_llm.invoke([sys] + state["messages"])]}

cap_builder = StateGraph(MessagesState)
cap_builder.add_node("copilot", copilot_node)
cap_builder.add_node("tools", ToolNode(capstone_tools))
cap_builder.add_edge(START, "copilot")

# TODO #3: Add conditional edges from "copilot" using `tools_condition`,
# AND add a normal edge from "tools" back to "copilot" so the agent reads tool outputs!
# <-- WRITE YOUR 2 GRAPH EDGE LINES HERE!

student_copilot = cap_builder.compile(checkpointer=MemorySaver())
print("✅ `student_copilot` compiled! Run the Automated Check cell below to test your work.")

In [ ]:
# @title 💡 Click to Show/Run Complete Capstone Solution
@tool
def needed_final_exam_score(current_weighted_avg: float, final_exam_weight: float, target_course_grade: float) -> str:
    """Calculate the exact percentage score needed on the final exam to achieve a target overall course grade, given the student's current average on completed work and the final exam weight."""
    w = final_exam_weight / 100.0 if final_exam_weight > 1.0 else final_exam_weight
    needed = (target_course_grade - current_weighted_avg * (1.0 - w)) / w
    return f"To achieve {target_course_grade}% overall (with final exam weight {w:.0%}), you need {needed:.1f}% on the final exam."

capstone_tools = [lookup_syllabus, needed_final_exam_score, send_extension_email]
capstone_llm = active_chat.bind_tools(capstone_tools)

def copilot_node(state: MessagesState):
    sys = SystemMessage(
        content=(
            "You are a supportive University Course Survival Copilot. "
            "Always check `lookup_syllabus` first if you need course policies or exam weights, "
            "use `needed_final_exam_score` for grade math, and use `send_extension_email` when asked to request an extension."
        )
    )
    return {"messages": [capstone_llm.invoke([sys] + state["messages"])]}

cap_builder = StateGraph(MessagesState)
cap_builder.add_node("copilot", copilot_node)
cap_builder.add_node("tools", ToolNode(capstone_tools))
cap_builder.add_edge(START, "copilot")
cap_builder.add_conditional_edges("copilot", tools_condition)
cap_builder.add_edge("tools", "copilot")

student_copilot = cap_builder.compile(checkpointer=MemorySaver())
print("✅ Complete Capstone Solution compiled into `student_copilot`!")

In [ ]:
# ==============================================================================
# 🧪 Automated Capstone Test Suite
# ==============================================================================
# Test 1: Verify your algebraic formula directly
math_out = needed_final_exam_score.invoke({
    "current_weighted_avg": 88.0,
    "final_exam_weight": 0.35,
    "target_course_grade": 90.0,
})
assert "93.7%" in math_out, f"❌ Formula check failed! Expected 93.7%, got: {math_out}"
print("✅ Test 1 Passed: Grade formula accurately computed 93.7%.")

# Test 2: Multi-step Syllabus RAG -> Grade Math Tool Chain
test_thread = {"configurable": {"thread_id": "capstone-grader-1"}}
agent_res = student_copilot.invoke(
    {"messages": [("user", "My current average is 88%. Check the syllabus for the final exam weight and tell me what score I need on the final exam to get a 90% overall.")]},
    config=test_thread,
)
used_tools = [m.name for m in agent_res["messages"] if m.type == "tool"]
assert "lookup_syllabus" in used_tools and "needed_final_exam_score" in used_tools, (
    f"❌ Expected agent to call BOTH `lookup_syllabus` and `needed_final_exam_score`, but it called: {used_tools}. "
    "Check your TODO #1a docstring and TODO #3 loop edge!"
)
print("✅ Test 2 Passed: Agent chained `lookup_syllabus` -> `needed_final_exam_score` autonomously!")
print("\n🤖 Copilot Response:\n", agent_res["messages"][-1].content)

In [ ]:
# @title 9.1 Interactive Course Copilot Playground (with Live Email HITL!)
import uuid

your_request = "I currently have an 86.5% average—can I still get a 4.0 (A) according to the syllabus? Also draft a 2-day homework extension email because I am presenting research at an out-of-state conference."  # @param {type:"string"}

live_thread = {"configurable": {"thread_id": f"live-{uuid.uuid4().hex[:6]}"}}
out = student_copilot.invoke({"messages": [("user", your_request)]}, config=live_thread)

# Check if the graph paused at our Human-in-the-Loop email interrupt!
snap = student_copilot.get_state(live_thread)
while snap.next:
    payload = snap.tasks[0].interrupts[0].value
    print("\n" + "=" * 60)
    print("⏸️ HUMAN-IN-THE-LOOP APPROVAL REQUIRED BEFORE SENDING EMAIL")
    print("=" * 60)
    print(payload["email_draft"])
    print("=" * 60)
    choice = input(payload["prompt"] + " ")
    out = student_copilot.invoke(Command(resume=choice), config=live_thread)
    snap = student_copilot.get_state(live_thread)

print("\n🎓 Final Copilot Response:\n")
print(out["messages"][-1].content)

## References & Further Resources

1. **[LangChain Academy](https://academy.langchain.com/courses/intro-to-langgraph)** – Free official courses & interactive notebooks  
   GitHub: <https://github.com/langchain-ai/langchain-academy>
2. **[LangGraph Documentation: Human-in-the-Loop & `interrupt`](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/wait-user-input/)** – Official guide to persistence, `MemorySaver`, and `Command(resume=...)`
3. **[LangChain Tutorials & Conceptual Guides](https://python.langchain.com/docs/tutorials/)** – Official documentation on LCEL, Tools, and RAG
4. **[LangGraph Cookbook](https://github.com/abhishekmaroon5/langgraph-cookbook)** – Practical multi-agent and cyclic graph recipes
5. **[Interrupt](https://interrupt.langchain.com/)** – The AI Agent Conference by LangChain
6. **[A Hands-On Guide to Building Intelligent Systems](https://docs.google.com/document/d/1rsaK53T3Lg5KoGwvf8ukOUvbELRtH-V0LnOIFDxBryE/preview?tab=t.0)** – Antonio Gulli (Agentic Design Patterns: Chaining, Routing, Reflection, Tool Use, HITL)
7. **[Prof. Mohammad Ghassemi – AI Agents Lectures](https://www.youtube.com/@ghassemi)** – Video lectures and tutorials on LLM systems